# Week 5 - Apache Spark Assignment

## Objective
To learn Apache Spark DataFrames by performing data loading, cleaning, transformation, filtering, aggregation, and analysis on a transactional dataset.

## Technologies Used
- Apache Spark
- PySpark
- Databricks
- Unity Catalog

## Dataset
A transactional dataset containing customer information, product details, sales, timestamps, and transaction records.

#### Q.1 What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

Apache Hadoop MapReduce is a distributed processing framework that processes data in two major phases: Map and Reduce. Although it is fault tolerant and scalable, it has several limitations that make it inefficient for modern analytics workloads

### Limitations of MapReduce

##### 1. Heavy Disk I/O:
MapReduce writes the output of every stage to disk before the next stage begins.
MapReduce writes intermediate results to HDFS(disk) after every Map and Reduce phase. 
For a 5-step algorithm, this means 10 read+write cycles.
Disk access is approximately *100× slower than RAM access*.

Read Data
->
Map
->
Write to Disk
->
Read Again
->
Reduce
->
Write Again

Frequent disk operations significantly increase execution time.

Spark fix: Intermediate data lives in RAM (via RDDs / DataFrames).  
Disk is only touched when memory is exhausted or output is explicitly saved.

##### 2. Poor Performance for Iterative Algorithms:
Machine Learning algorithms repeatedly process the same dataset.

Iteration 1
->
Read Disk
->
Iteration 2
->
Read Disk Again
->
Iteration 3
->
Read Disk Again

Repeated disk reads make ML workloads slow.

Spark fix: df.cache() / df.persist() keeps the dataset in RAM.  
Subsequent iterations read from memory → 10–100× speed-up.

##### 3. High Latency:
Even small analytical queries require an entire MapReduce job.
Therefore interactive analytics is inefficient

Spark fix: Lazy evaluation + in-memory execution → sub-second  
interactive queries via Spark SQL and the DataFrame API.

##### 4. Rigid Programming Model: 
MapReduce follows a fixed execution model.

Structure of code and sequence is very rigid. 
If we want to perform filter operations we have to perform: 
filter -> map -> shuffle -> reduce

Complex ETL pipelines require a lot of bollerplate

Spark fix:  
Rich, flexible API — `filter()`, `groupBy()`, `join()`, `window()` — 
all expressed naturally. Spark's Catalyst Optimizer automatically finds 
the best execution plan.

##### 5. Only Batch Processing Supported: 
Real time processing is not supported, so we cannot work on streaming data.

Spark fix: Spark Structured Streaming uses the same DataFrame API 
for both batch and real-time — one codebase, one engine

##### 6. No Built-in Libraries: 
MapReduce doesn't provide integrated support for: 
1. SQL
2. Streaming
3. ML
4. Graph Analytics

Spark fix: Spark includes all these features at one platform

##### 7. No Interactive Model/Way to Monitor: 
MapReduce ships code as JAR packages with minimal visibility into execution.  
There is no built-in web UI to inspect running jobs, stage timings, 
or memory usage in real time.

Spark fix:  
Spark UI (port 4040) provides real-time visibility into:
- DAG execution stages
- Task-level timing and metrics
- Memory and shuffle statistics
- Failed task diagnostics

##### 8. High Network Communication
During the Shuffle phase, MapReduce transfers large amounts of intermediate data across the cluster.
This increases:
• Network congestion
• Execution latency
• Cluster resource utilization

Spark fixes:
- Predicate Pushdown
- Broadcast Joins
- Adaptive Query Execution (AQE)



#### Internal Working: 

##### MapReduce: 

Read Data
->
Map
->
Write Intermediate Data to HDFS
->
Read Again
->
Reduce
->
Write Final Output

##### Spark: 

Read Data
->
Logical Plan
->
Catalyst Optimization
->
Physical Plan
->
DAG Scheduler
->
Execute
->
Result

#### Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems. 

In-Memory Computing is one of Apache Spark core architectural features that significantly improves the performance of iterative algorithms such as Machine Learning, Graph Processing, and Interactive Analytics.

Unlike Hadoop MapReduce, which writes intermediate results to disk after every processing stage, Spark stores intermediate data in memory (RAM) whenever possible. This minimizes expensive disk read/write operations and allows repeated computations to reuse cached datasets.

As a result, iterative algorithms execute much faster because the same dataset is accessed directly from memory instead of being reloaded from disk in every iteration

### How MapReduce Works:
For every iteration:
- - - - 
Read Dataset from HDFS -> Process Data -> Write Intermediate Output to HDFS -> Read Again for Next Iteration -> Process Again -> Write Again
- - - - 

###### Example

Suppose we train a Machine Learning model using Gradient Descent for 10 iterations.
MapReduce performs:
- - - - - 
Iteration 1
Read Disk → Process → Write Disk

Iteration 2
Read Disk → Process → Write Disk

Iteration 3
Read Disk → Process → Write Disk

...

Iteration 10
Read Disk → Process → Write Disk
- - - - - -
This means the same dataset is read from disk 10 times, resulting in high disk I/O and longer execution time

#### How Spark Works:
- - - - -  
Read Dataset Once -> Store Dataset in RAM -> Iteration 1 -> Iteration 2 -> Iteration 3 ... -> Final Output Written to Disk
- - - - - -
Spark reads the dataset only once. If the dataset is cached or persisted, all subsequent iterations reuse the in-memory copy, eliminating repeated disk access

Spark stores cached DataFrames as partitions across the memory of worker nodes.
- - - - - - - - - -
Cluster

Worker 1
Partition 1

Worker 2
Partition 2

Worker 3
Partition 3

Worker 4
Partition 4
- - - - - - - - - 
Each executor accesses its local partition directly from RAM instead of reading it from disk, enabling parallel and efficient processing.

### if RAM Becomes Full:

Spark does not fail immediately.

Instead, it follows this strategy:

Store as much data as possible in RAM.
If memory becomes insufficient, spill the remaining data to disk.
Continue processing without data loss.

WE can control this behavior using:

df.persist(StorageLevel.MEMORY_AND_DISK)

This stores data in memory first and automatically spills excess data to disk if required.


| Feature              | Hadoop MapReduce | Apache Spark                        |
| -------------------- | ---------------- | ----------------------------------- |
| Intermediate Storage | Disk (HDFS)      | Memory (RAM)                        |
| Dataset Reads        | Every Iteration  | Once (if cached)                    |
| Machine Learning     | Slow             | Fast                                |
| Interactive Queries  | High Latency     | Low Latency                         |
| Disk I/O             | High             | Very Low                            |
| Performance          | Slower           | 10–100× Faster (workload-dependent) |


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Verify the spark session 
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("Week5Assignment").getOrCreate()

In [0]:
# Loading data which is present in Unity Catalog Volume

df = (
    spark.read
    .format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .option("mode","FAILFAST")
    .load("/Volumes/dbacademy/default/tutorials/week5.csv")
)

In [0]:
# Initial data exploration 

df.show(5)
df.printSchema()

+--------------+-------+----------------+-------+----------------+-----------+-------+-----------+-----+----+------------+-------------------+--------------------+--------+-------+--------+--------+------------+------------+----------------+
|transaction_id|user_id|transaction_date| region|product_category|sale_amount| status|       city|state| age|subscription|      raw_timestamp|               email|username|  price|quantity|store_id|product_name|discount_pct|customer_segment|
+--------------+-------+----------------+-------+----------------+-----------+-------+-----------+-----+----+------------+-------------------+--------------------+--------+-------+--------+--------+------------+------------+----------------+
|       T000522| U00042|      2024-06-25|   East|           Books|     343.96| Active|Los Angeles|   WA|60.0|    Standard|2023-03-02 15:15:00|useru00042@yahoo.com|user_318| 369.57|       6|    S008| Product_161|         0.3|     Home Office|
|       T000738| U00498|      20

In [0]:
print("Total Rows:",df.count())
print("Total Columns:",len(df.columns))
print(df.columns)
df.dtypes

Total Rows: 1000
Total Columns: 20
['transaction_id', 'user_id', 'transaction_date', 'region', 'product_category', 'sale_amount', 'status', 'city', 'state', 'age', 'subscription', 'raw_timestamp', 'email', 'username', 'price', 'quantity', 'store_id', 'product_name', 'discount_pct', 'customer_segment']


[('transaction_id', 'string'),
 ('user_id', 'string'),
 ('transaction_date', 'date'),
 ('region', 'string'),
 ('product_category', 'string'),
 ('sale_amount', 'double'),
 ('status', 'string'),
 ('city', 'string'),
 ('state', 'string'),
 ('age', 'double'),
 ('subscription', 'string'),
 ('raw_timestamp', 'string'),
 ('email', 'string'),
 ('username', 'string'),
 ('price', 'double'),
 ('quantity', 'int'),
 ('store_id', 'string'),
 ('product_name', 'string'),
 ('discount_pct', 'double'),
 ('customer_segment', 'string')]

In [0]:
missing_val = df.select([
    count(when(col(c).isNull(),c)).alias(c) 
    for c in df.columns 
])
display(missing_val)

transaction_id,user_id,transaction_date,region,product_category,sale_amount,status,city,state,age,subscription,raw_timestamp,email,username,price,quantity,store_id,product_name,discount_pct,customer_segment
0,0,0,0,0,57,92,0,0,54,0,40,57,74,74,0,0,0,94,0


In [0]:
# Duplicate records
print("Duplicate Rows:",df.count() - df.dropDuplicates().count())

Duplicate Rows: 50


#### Dataset Observations

After exploring the dataset, the following observations were made:

* The dataset contains **1000 records** and **20 columns**.
* **50 duplicate records** are present and need to be removed before analysis.
* Missing values are found in multiple columns, including **sale_amount, status, age, raw_timestamp, email, username, price, and discount_pct**.
* The **transaction_date** column has already been inferred as `DateType`.
* The **raw_timestamp** column is currently stored as a **StringType** and requires conversion to `TimestampType`.
* Numeric columns such as **sale_amount**, **price**, and **quantity** are already inferred correctly and are suitable for analytical operations.


In [0]:
## Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date. 

# Total records before removing duplicates
print("Total Records Before Cleaning :", df.count())

# Remove duplicate records based on user_id and transaction_date
df_deduplicated = df.dropDuplicates(["user_id", "transaction_date"])

print("Total Records After Removing Duplicates :", df_deduplicated.count())
print("Duplicate Records Removed :", df.count() - df_deduplicated.count())
display(df_deduplicated)

Total Records Before Cleaning : 1000
Total Records After Removing Duplicates : 949
Duplicate Records Removed : 51


transaction_id,user_id,transaction_date,region,product_category,sale_amount,status,city,state,age,subscription,raw_timestamp,email,username,price,quantity,store_id,product_name,discount_pct,customer_segment
T000618,U00442,2024-10-06,East,Toys,3213.95,Active,Reno,AZ,29.0,Premium,2024-05-22T14:28:00,null,user_230,1816.42,7,S011,Product_067,0.3,Home Office
T000068,U00096,2023-11-19,South,Electronics,4328.96,Inactive,Denver,IL,67.0,Basic,2024-10-30T15:26:00,useru00096@outlook.com,user_251,1415.67,4,S005,Product_017,0.14,Consumer
T000529,U00389,2024-08-06,Central,Books,4166.32,Churned,Los Angeles,WA,38.0,Basic,08/24/2023 18:56:00,useru00389@gmail.com,user_111,2151.59,12,S007,Product_168,0.23,Consumer
T000261,U00363,2024-03-28,Central,Toys,3665.43,Pending,Denver,AZ,67.0,Standard,2024-03-25 13:49:00,useru00363@yahoo.com,user_259,668.71,7,S006,Product_126,0.08,Corporate
T000045,U00354,2023-05-01,Central,Books,1844.04,null,Seattle,CO,24.0,Free,2024-01-18T13:44:00,useru00354@outlook.com,,622.43,14,S013,Product_042,0.26,Corporate
T000803,U00001,2023-08-02,Central,Electronics,null,Inactive,Reno,IL,65.0,Standard,2024-02-17T14:15:00,null,user_267,1946.47,2,S005,Product_061,null,Home Office
T000496,U00387,2024-08-28,West,Clothing,3320.93,Inactive,Portland,NV,59.0,Basic,07/13/2024 20:12:00,useru00387@yahoo.com,user_150,1068.67,2,S008,Product_013,0.25,Corporate
T000656,U00224,2023-06-02,West,Clothing,2627.83,Pending,Phoenix,CO,null,Free,2024-02-17 13:27:00,useru00224@yahoo.com,user_13,null,10,S008,Product_240,0.06,Consumer
T000306,U00080,2023-07-14,West,Beauty,2462.35,null,New York,AZ,39.0,Basic,2024-08-25 02:26:00,useru00080@gmail.com,user_290,1908.56,13,S015,Product_179,0.25,Home Office
T000034,U00067,2024-01-15,East,Clothing,1602.39,Active,Houston,IL,24.0,Premium,2024-01-29 19:26:00,useru00067@gmail.com,user_296,478.64,9,S008,Product_256,0.44,Consumer


### Observation

* The dataset initially contained **1000 records**.
* After removing duplicates based on **`user_id`** and **`transaction_date`**, **949 records** remained.
* A total of **51 duplicate records** were removed.

**Note:** This count differs from removing completely identical rows because `dropDuplicates(["user_id", "transaction_date"])` identifies duplicates based only on the specified columns, regardless of differences in other columns.


In [0]:
## Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount. 

from pyspark.sql.functions import avg

west_sales = (
    df.filter(col("region") == "West")
      .groupBy("product_category")
      .agg( avg("sale_amount").alias("Average_Sale_Amount"))
      .orderBy(col("Average_Sale_Amount").desc())
)
display(west_sales)

print("Total Records in West Region :", df.filter(col("region")=="West").count())

product_category,Average_Sale_Amount
Sports,2645.7877777777776
Home & Kitchen,2559.3860000000004
Beauty,2460.1923529411756
Furniture,2424.852894736842
Books,2330.775
Toys,2202.656428571429
Clothing,2154.6864864864856
Electronics,2044.4197368421055


Total Records in West Region : 312


#### Observation

* The dataset contains **8 product categories** in the **West** region.
* **Sports** has the highest average sale amount (**2645.79**), indicating strong sales performance in this region.
* **Electronics** has the lowest average sale amount (**2044.42**).
* The results are sorted in descending order of average sales, making it easier to identify the best-performing categories.
* This type of analysis helps businesses understand which product categories generate higher revenue in a specific region and supports inventory planning and marketing decisions.




### Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

Handling missing values is an important step in data cleaning because null values can lead to incorrect analysis, aggregation errors, or poor machine learning model performance.

Apache Spark provides two commonly used methods for handling missing values:

#### 1. .na.drop()
Removes rows containing null values.
Can remove rows based on all columns or selected columns.
Best used when records with missing values are not useful for analysis.
#### 2. .na.fill()
Replaces null values with a specified value instead of removing the entire row.
Helps preserve valuable data while handling missing information.
Different replacement values can be used for different columns.

In this dataset, the status column contains 92 missing values. Instead of deleting these records, the missing values are replaced with "Unknown", ensuring that no transaction data is lost.

In [0]:
from pyspark.sql.functions import col, when, count

# Count null values before filling
null_before = df.filter(col("status").isNull()).count()

print("Null values before filling :", null_before)

# Fill missing values
df_filled = df.na.fill({"status": "Unknown"})

# Count null values after filling
null_after = df_filled.filter(col("status").isNull()).count()

print("Null values after filling :", null_after)

# Display sample records
display(
    df_filled.select("transaction_id", "status").limit(10)
)

df_filled.groupBy("status").count().orderBy("count", ascending=False).show()

Null values before filling : 92
Null values after filling : 0


transaction_id,status
T000522,Active
T000738,Active
T000741,Active
T000661,Churned
T000412,Churned
T000679,Active
T000627,Active
T000514,Pending
T000860,Inactive
T000137,Unknown


+--------+-----+
|  status|count|
+--------+-----+
|  Active|  416|
|Inactive|  193|
| Pending|  155|
| Churned|  144|
| Unknown|   92|
+--------+-----+



##### Observation

* The **status** column initially contained **92 null values**.
* After applying **`.na.fill({"status":"Unknown"})`**, all missing values were successfully replaced.
* No rows were removed, preserving the complete dataset for further analysis.
* Replacing missing categorical values with **"Unknown"** is often preferred over deleting records because it prevents unnecessary data loss while clearly identifying missing information.


In [0]:
## Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100. 

from pyspark.sql.functions import count, col

city_count = (
    df.groupBy("city")
      .agg(count("*").alias("Total_Records"))
      .filter(col("Total_Records") > 100)
      .orderBy(col("Total_Records").desc())
)

display(city_count)

print("Total Unique Cities :", df.select("city").distinct().count())

city,Total_Records
New York,134
Los Angeles,134
Chicago,125
Phoenix,123
Houston,116


Total Unique Cities : 15


##### Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them? 

Apache Spark DataFrames are immutable, which means that once a DataFrame is created, its data cannot be modified directly.

Any data cleaning operation such as dropping columns, renaming columns, filtering rows, or filling missing values does not change the original DataFrame. Instead, Spark creates a new DataFrame containing the required modifications while leaving the original DataFrame unchanged.

This design improves fault tolerance, enables lazy evaluation, and allows Spark to optimize execution using the Catalyst Optimizer before running the job.

In this assignment, operations such as dropDuplicates(), na.fill(), and groupBy() all created new DataFrames rather than modifying the original dataset.

In [0]:
print("Immutability of Spark DataFrames")

print("\nOriginal Columns:")
print(df.columns)

temp1 = df.drop("discount_pct")
temp2 = df.withColumnRenamed("age", "customer_age")

print("\nChecking Original DataFrame")

print("'discount_pct' in df :", "discount_pct" in df.columns)
print("'discount_pct' in temp1 :", "discount_pct" in temp1.columns)

print("'age' in df :", "age" in df.columns)
print("'customer_age' in temp2 :", "customer_age" in temp2.columns)

df_cleaned = (
    df
    .drop("discount_pct")
    .withColumnRenamed("age", "customer_age")
    .withColumnRenamed("sale_amount", "total_sale")
    .withColumn("price", col("price").cast(DoubleType()))
)

print("\nCleaned DataFrame Columns:")
print(df_cleaned.columns)

print("\nDataFrame IDs")
print("Original df :", id(df))
print("Cleaned df  :", id(df_cleaned))

print("\nSchema Comparison")
print("Original price type :", dict(df.dtypes)["price"])
print("Cleaned price type  :", dict(df_cleaned.dtypes)["price"])

print("\nSample Data")
df_cleaned.select(
    "customer_age",
    "total_sale",
    "price"
).show(5)

print("\nObservation")
print("- Original DataFrame remained unchanged.")
print("- Spark created a new DataFrame after every transformation.")
print("- DataFrames are immutable.")
print("- Immutability enables Lazy Evaluation, Fault Tolerance and Catalyst Optimization.")

Immutability of Spark DataFrames

Original Columns:
['transaction_id', 'user_id', 'transaction_date', 'region', 'product_category', 'sale_amount', 'status', 'city', 'state', 'age', 'subscription', 'raw_timestamp', 'email', 'username', 'price', 'quantity', 'store_id', 'product_name', 'discount_pct', 'customer_segment']

Checking Original DataFrame
'discount_pct' in df : True
'discount_pct' in temp1 : False
'age' in df : True
'customer_age' in temp2 : True

Cleaned DataFrame Columns:
['transaction_id', 'user_id', 'transaction_date', 'region', 'product_category', 'total_sale', 'status', 'city', 'state', 'customer_age', 'subscription', 'raw_timestamp', 'email', 'username', 'price', 'quantity', 'store_id', 'product_name', 'customer_segment']

DataFrame IDs
Original df : 280476602238256
Cleaned df  : 280475997390896

Schema Comparison
Original price type : double
Cleaned price type  : double

Sample Data
+------------+----------+-------+
|customer_age|total_sale|  price|
+------------+------

In [0]:
## Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'. 

from pyspark.sql.functions import col

premium_customers = (
    df.filter(
        (col("age").between(18, 30)) &
        (col("subscription") == "Premium")
    )
)

# Display filtered records
display(premium_customers)

# Count the filtered records
print("Total Premium Customers (Age 18–30):", premium_customers.count())

transaction_id,user_id,transaction_date,region,product_category,sale_amount,status,city,state,age,subscription,raw_timestamp,email,username,price,quantity,store_id,product_name,discount_pct,customer_segment
T000412,U00489,2023-05-27,West,Sports,2158.38,Churned,New York,AZ,21.0,Premium,10/22/2023 08:31:00,useru00489@gmail.com,user_133,1884.9,11,S005,Product_073,0.05,Home Office
T000627,U00146,2023-12-22,East,Toys,4812.53,Active,Los Angeles,CO,20.0,Premium,07/11/2024 12:28:00,useru00146@outlook.com,user_337,2335.09,7,S013,Product_040,null,Home Office
T000514,U00299,2024-10-03,South,Toys,3158.7,Pending,New York,NV,22.0,Premium,2023-03-25T05:05:00,useru00299@gmail.com,user_352,null,6,S011,Product_272,0.1,Home Office
T000528,U00137,2024-07-23,East,Furniture,807.72,Pending,Denver,AZ,27.0,Premium,null,useru00137@outlook.com,user_344,1017.2,1,S007,Product_055,0.37,Consumer
T000622,U00241,2024-07-01,West,Home & Kitchen,4526.16,Inactive,Austin,GA,25.0,Premium,06/15/2023 20:05:00,useru00241@outlook.com,user_94,487.5,7,S003,Product_294,0.27,Corporate
T000300,U00288,2024-10-03,West,Sports,4672.29,Inactive,New York,IL,28.0,Premium,null,useru00288@gmail.com,user_34,699.75,1,S002,Product_112,0.21,Corporate
T000618,U00442,2024-10-06,East,Toys,3213.95,Active,Reno,AZ,29.0,Premium,2024-05-22T14:28:00,null,user_230,1816.42,7,S011,Product_067,0.3,Home Office
T000067,U00181,2024-04-13,Central,Books,4594.23,Pending,New York,AZ,22.0,Premium,2024-01-18T22:09:00,useru00181@outlook.com,user_197,1219.24,15,S011,Product_089,0.28,Consumer
T000089,U00486,2023-07-12,South,Toys,2715.2,Active,Houston,CA,29.0,Premium,2024-05-04T00:21:00,useru00486@gmail.com,user_91,2111.49,1,S003,Product_073,0.39,Corporate
T000238,U00015,2023-09-09,Central,Books,2326.16,Inactive,Los Angeles,CA,27.0,Premium,2023-09-22T08:42:00,useru00015@gmail.com,user_249,143.04,6,S010,Product_153,0.43,Corporate


Total Premium Customers (Age 18–30): 89


#### Spark Internal Concept

`filter()` is a **Narrow Transformation**.

Each partition is processed independently without exchanging data with other partitions.

Since no shuffle occurs, filtering is one of the fastest operations in Apache Spark.

-> The between() function is used to define the age range, while the logical AND (&) operator ensures that both conditions are satisfied simultaneously.


#### Q9. When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like `sum()` or `avg()`?

Handling missing values before performing mathematical aggregations is an important step in data cleaning because null values can affect the accuracy and interpretation of analytical results.

In Apache Spark, aggregation functions such as **`sum()`** and **`avg()`** ignore null values during computation. While this prevents runtime errors, it may produce misleading results because the calculations are performed only on the available (non-null) records.

During the exploratory analysis of this dataset, the **`sale_amount`** column was found to contain **57 null values**. If these missing values are not handled explicitly, aggregation functions will silently exclude those records, reducing the effective sample size and potentially producing biased business insights.

Depending on business requirements, missing values can be handled by:

* Removing incomplete records using `.na.drop()`
* Replacing missing values using `.na.fill()`
* Imputing statistical values such as mean or median

Explicitly handling null values makes the data cleaning process transparent, reproducible, and suitable for reliable analytics.

In [0]:
from pyspark.sql.functions import col, sum, avg, count

null_count = df.filter(col("sale_amount").isNull()).count()

print("Null values in sale_amount:", null_count)

print("Before Handling Null Values")
df.select(
    sum("sale_amount").alias("Total_Sales"),
    avg("sale_amount").alias("Average_Sales"),
    count("sale_amount").alias("Non_Null_Records"),
    count("*").alias("Total_Records")
).show()

df_filled = df.na.fill({"sale_amount": 0})

print("After Filling Null Values")
df_filled.select(
    sum("sale_amount").alias("Total_Sales"),
    avg("sale_amount").alias("Average_Sales"),
    count("sale_amount").alias("Non_Null_Records"),
    count("*").alias("Total_Records")
).show()

Null values in sale_amount: 57
Before Handling Null Values
+------------------+-----------------+----------------+-------------+
|       Total_Sales|    Average_Sales|Non_Null_Records|Total_Records|
+------------------+-----------------+----------------+-------------+
|2319701.2800000007|2459.916521739131|             943|         1000|
+------------------+-----------------+----------------+-------------+

After Filling Null Values
+------------------+------------------+----------------+-------------+
|       Total_Sales|     Average_Sales|Non_Null_Records|Total_Records|
+------------------+------------------+----------------+-------------+
|2319701.2800000007|2319.7012800000007|            1000|         1000|
+------------------+------------------+----------------+-------------+



In [0]:
## Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

#  The `raw_timestamp` column is currently stored as a string. Before performing time-based analysis, it should be converted to `TimestampType`. After conversion, the column is renamed to `event_time` for better readability and consistency

display(df.select("raw_timestamp").limit(20))

from pyspark.sql.functions import col, regexp_replace, when, to_timestamp

df_timestamp = (
    df
    # Replace 'T' with space
    .withColumn(
        "raw_timestamp",
        regexp_replace(col("raw_timestamp"), "T", " ")
    )

    # Convert timestamps based on detected format
    .withColumn(
        "event_time",
        when(
            col("raw_timestamp").rlike("^[0-9]{2}/[0-9]{2}/[0-9]{4}"),
            to_timestamp(col("raw_timestamp"), "MM/dd/yyyy HH:mm:ss")
        ).otherwise(
            to_timestamp(col("raw_timestamp"), "yyyy-MM-dd HH:mm:ss")
        )
    )

    # Remove old column
    .drop("raw_timestamp")
)

display(df_timestamp.select("event_time").limit(10))
df_timestamp.printSchema()

raw_timestamp
2023-03-02 15:15:00
2024-02-10T18:59:00
06/25/2024 22:38:00
2023-11-08 17:08:00
10/22/2023 08:31:00
2023-08-23 10:54:00
07/11/2024 12:28:00
2023-03-25T05:05:00
2024-12-20 10:56:00
07/24/2024 15:11:00


event_time
2023-03-02T15:15:00.000Z
2024-02-10T18:59:00.000Z
2024-06-25T22:38:00.000Z
2023-11-08T17:08:00.000Z
2023-10-22T08:31:00.000Z
2023-08-23T10:54:00.000Z
2024-07-11T12:28:00.000Z
2023-03-25T05:05:00.000Z
2024-12-20T10:56:00.000Z
2024-07-24T15:11:00.000Z


root
 |-- transaction_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: double (nullable = true)
 |-- subscription: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



#### Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation? 

A **Shuffle** is the process of redistributing data across different partitions so that records with the same key are brought together for operations such as `groupBy()`, `join()`, `distinct()`, and `reduceByKey()`.

A **wide transformation** is one in which an output partition depends on multiple input partitions. Since data has to move across the cluster, network communication occurs, making the operation more expensive than narrow transformations.

##### Narrow vs Wide Transformation

| Narrow Transformation | Wide Transformation |
|----------------------|---------------------|
| Data remains in the same partition | Data is redistributed across partitions |
| No network communication | Requires network communication (Shuffle) |
| Faster execution | Slower execution |
| Examples: `filter()`, `select()`, `withColumn()` | Examples: `groupBy()`, `join()`, `distinct()` |

##### Working of Shuffle in `groupBy()`

Partition 1
```
West 500
East 300
West 200
```

Partition 2
```
East 400
West 100
South 600
```
↓

**Shuffle**

↓

```
Partition A
West → 500 + 200 + 100 = 800
```
```
Partition B
East → 300 + 400 = 700
```

```
Partition C
South → 600
```

During the shuffle phase, Spark transfers records having the same key to the same partition, after which the aggregation is performed.

##### Why is Shuffle Expensive?

- Requires network data transfer between worker nodes.
- Creates temporary shuffle files on disk.
- Involves serialization and deserialization of data.
- Increases execution time compared to narrow transformations.

In [0]:
from pyspark.sql.functions import sum

# Group data by region
region_sales = (
    df.groupBy("region")
      .agg(sum("sale_amount").alias("Total_Sales"))
)
display(region_sales)

region,Total_Sales
East,620608.3499999994
South,435696.8999999998
Central,577664.55
West,685731.480000001


##### Observation

The `groupBy()` operation triggered a Shuffle because records belonging to the same region were located in different partitions. Spark redistributed the data across the cluster, grouped identical keys together, and then computed the final aggregation. This data movement makes `groupBy()` a wide transformation and one of the most expensive operations in Spark.

Performance Tip: Since Shuffle is expensive, it should be minimized whenever possible. Spark optimizations such as Catalyst Optimizer, Adaptive Query Execution (AQE), and appropriate partitioning strategies help reduce unnecessary Shuffle operations and improve query performance.

In [0]:
## Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string. 

from pyspark.sql import functions as F
print("Before Cleaning")

print(f"Total Records:{df.count()}")
print(f"NULL Emails:{df.filter(F.col('email').isNull()).count()}")
print(f"NULL Usernames: {df.filter(F.col('username').isNull()).count()}")
print(f"Empty Usernames: {df.filter(F.trim(F.col('username')) == '').count()}")

#IDENTIFY INVALID RECORDS 
df_bad = df.filter( F.col("email").isNull() |
    F.col("username").isNull() |
    (F.trim(F.col("username")) == "")
)

print(f"\nInvalid Records Found : {df_bad.count()}")

display(
    df_bad.select( "transaction_id", "user_id", "email", "username")
)

# REMOVE INVALID RECORDS 
df_clean = df.filter( F.col("email").isNotNull() &
    F.col("username").isNotNull() &
    (F.trim(F.col("username")) != "")
)

print("After Cleaning")


print(f"Total Records: {df_clean.count()}")
print(f"Records Removed: {df.count() - df_clean.count()}")

print(f"Remaining NULL Emails: {df_clean.filter(F.col('email').isNull()).count()}")
print(f"Remaining NULL Usernames: {df_clean.filter(F.col('username').isNull()).count()}")
print(f"Remaining Empty Usernames: {df_clean.filter(F.trim(F.col('username')) == '').count()}")

# Display cleaned dataset
display(df_clean.limit(10))

Before Cleaning
Total Records:1000
NULL Emails:57
NULL Usernames: 74
Empty Usernames: 18

Invalid Records Found : 147


transaction_id,user_id,email,username
T000679,U00169,null,user_359
T000281,U00155,useru00155@yahoo.com,null
T000500,U00194,useru00194@gmail.com,null
T000585,U00194,null,user_2
T000278,U00133,null,user_166
T000618,U00442,null,user_230
T000693,U00358,useru00358@gmail.com,null
T000494,U00256,useru00256@gmail.com,null
T000590,U00110,useru00110@gmail.com,null
T000523,U00097,null,user_334


After Cleaning
Total Records: 853
Records Removed: 147
Remaining NULL Emails: 0
Remaining NULL Usernames: 0
Remaining Empty Usernames: 0


transaction_id,user_id,transaction_date,region,product_category,sale_amount,status,city,state,age,subscription,raw_timestamp,email,username,price,quantity,store_id,product_name,discount_pct,customer_segment
T000522,U00042,2024-06-25,East,Books,343.96,Active,Los Angeles,WA,60.0,Standard,2023-03-02 15:15:00,useru00042@yahoo.com,user_318,369.57,6,S008,Product_161,0.3,Home Office
T000738,U00498,2023-12-05,Central,Books,3729.19,Active,Los Angeles,NY,65.0,Basic,2024-02-10T18:59:00,useru00498@outlook.com,user_362,649.9,5,S004,Product_295,0.44,Corporate
T000741,U00028,2023-01-29,West,Beauty,542.11,Active,Chicago,CO,36.0,Standard,06/25/2024 22:38:00,useru00028@outlook.com,user_347,2409.79,7,S013,Product_041,0.46,Consumer
T000661,U00500,2024-03-31,South,Books,4639.74,Churned,Los Angeles,NY,49.0,Standard,2023-11-08 17:08:00,useru00500@yahoo.com,user_183,1844.93,8,S009,Product_192,null,Consumer
T000412,U00489,2023-05-27,West,Sports,2158.38,Churned,New York,AZ,21.0,Premium,10/22/2023 08:31:00,useru00489@gmail.com,user_133,1884.9,11,S005,Product_073,0.05,Home Office
T000627,U00146,2023-12-22,East,Toys,4812.53,Active,Los Angeles,CO,20.0,Premium,07/11/2024 12:28:00,useru00146@outlook.com,user_337,2335.09,7,S013,Product_040,null,Home Office
T000514,U00299,2024-10-03,South,Toys,3158.7,Pending,New York,NV,22.0,Premium,2023-03-25T05:05:00,useru00299@gmail.com,user_352,null,6,S011,Product_272,0.1,Home Office
T000860,U00298,2023-07-01,Central,Clothing,3367.79,Inactive,Phoenix,TX,46.0,Premium,2024-12-20 10:56:00,useru00298@gmail.com,user_235,2305.53,14,S014,Product_158,0.31,Home Office
T000137,U00300,2023-09-17,West,Furniture,2827.7,null,Los Angeles,OR,63.0,Basic,07/24/2024 15:11:00,useru00300@yahoo.com,user_158,1936.62,3,S014,Product_061,0.37,Home Office
T000812,U00202,2024-04-08,South,Furniture,4238.05,Active,Los Angeles,AZ,59.0,Free,2023-05-14T08:04:00,useru00202@yahoo.com,user_227,2111.55,5,S008,Product_240,0.41,Home Office


#### Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column? 

The `.agg()` function in Apache Spark is used to perform one or more aggregate calculations in a single operation. Instead of scanning the dataset multiple times for different statistics, Spark computes all required aggregations together, making the operation more efficient.

In this task, the `price` column is analyzed to calculate:
- Minimum Price
- Maximum Price
- Average Price

Using `.agg()` improves readability and performance by combining multiple aggregate functions into a single query.



In [0]:
from pyspark.sql.functions import min, max, avg

# Calculate multiple statistics
price_statistics = df.agg(
    min("price").alias("Minimum_Price"),
    max("price").alias("Maximum_Price"),
    avg("price").alias("Average_Price")
)

# Display the result
display(price_statistics)

print("Schema of Aggregated DataFrame:")
price_statistics.printSchema()

Minimum_Price,Maximum_Price,Average_Price
11.83,2490.59,1275.912829373651


Schema of Aggregated DataFrame:
root
 |-- Minimum_Price: double (nullable = true)
 |-- Maximum_Price: double (nullable = true)
 |-- Average_Price: double (nullable = true)



##### Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats? 

1. **Silent fallback to StringType** — if Spark can't confidently match a consistent date/timestamp pattern across the sampled rows (exactly the situation from Q10, where the same column has three different formats), it gives up and infers the column as a plain string instead of a date or timestamp type. No error is raised — the column just isn't usable for date arithmetic until you cast it manually. 
2. **Inference is based on a sample, not the full dataset** — Spark doesn't necessarily scan every row to infer schema. If the sampled rows happen to look like clean dates but other rows further down don't, the inferred type may not actually fit the whole column. 
3. **Misinterpreted formats produce wrong values, not errors** — this is the most dangerous case. A date like 03/04/2024 is ambiguous: is it March 4th or April 3rd? If Spark infers a format assumption (e.g., MM/dd/yyyy) but some rows are actually dd/MM/yyyy, the dates get parsed into a valid-looking but *incorrect* date — no null, no crash, just silently wrong data that's hard to catch downstream. 
4. **Inconsistent typing across runs** — since inference depends on sampling, re-reading the same file (especially if it's appended to or reordered) could in principle infer a different schema, making pipelines fragile and non-reproducible. 

**The safer practice:** read date/timestamp columns as StringType explicitly (or define an explicit schema with StructType), then parse them deliberately using to_timestamp() with known format strings — as in Q10 — so format mismatches are visible and handled, rather than silently guessed at.


In [0]:
## Quick code illustration 

# Demonstrate: inferSchema gives up and falls back to StringType
# when a column has mixed date formats

df_inferred = spark.read.csv(
    "/Volumes/dbacademy/default/tutorials/week5.csv",
    header=True,
    inferSchema=True
)

df_inferred.printSchema()
# raw_timestamp likely shows up as: -- raw_timestamp: string (nullable = true)
# even though it "looks like" dates to a human reading the file



root
 |-- transaction_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: double (nullable = true)
 |-- subscription: string (nullable = true)
 |-- raw_timestamp: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- customer_segment: string (nullable = true)



**Observation:** Even though every value in `raw_timestamp` is technically a
date, Spark infers it as `string` because the formats aren't consistent enough
for its inference logic to commit to a date/timestamp type. This is exactly
why the explicit `to_timestamp()` + `coalesce()` approach from Q10 was
necessary — `inferSchema=True` alone would never have produced a usable
timestamp column here.

#### Q15: Write a final processing pipeline that: 
Filters out duplicates.
Fills null prices with 0. 
Groups by store_id to calculate total revenue. 

##### Objective
 Build a Spark ETL pipeline that performs the following operations: 
 1. Remove duplicate records. 
 2. Replace NULL values in the `price` column with **0**. 
 3. Group the cleaned data by **store_id**. 
 4. Calculate the **Total Revenue** for each store. 
 
##### Explanation
This pipeline demonstrates a simple ETL workflow in Apache Spark. 
- Duplicate records are removed to improve data quality. 
- Missing values in the `price` column are replaced with **0** before aggregation. 
- Revenue for each transaction is calculated as: 

> **Revenue = Price × Quantity**

- Finally, the dataset is grouped by **store_id** to calculate the total revenue generated by each store. 
Using method chaining allows Spark to optimize the complete execution plan through **Catalyst Optimizer** and execute the pipeline efficiently using **lazy evaluation**.

In [0]:
from pyspark.sql.functions import col, sum

store_revenue = (
    df
    .dropDuplicates(["transaction_id"])
    .na.fill({"price": 0})
    .withColumn(
        "revenue",
        col("price") * col("quantity")
    )
    .groupBy("store_id")
    .agg(
        sum("revenue").alias("Total_Revenue")
    )
    .orderBy(col("Total_Revenue").desc())
)

display(store_revenue)

store_id,Total_Revenue
S003,765271.92
S005,713188.8300000002
S014,665210.18
S013,648298.88
S002,617001.2400000002
S015,614183.0400000002
S012,611093.6400000001
S007,593567.12
S011,586421.4099999999
S001,579714.37


In [0]:
print("Pipeline Validation:")

print(f"Original Records: {df.count()}")

print(f"After Removing Duplicates: "
      f"{df.dropDuplicates(['transaction_id']).count()}")

print(f"Remaining NULL Prices: "
      f"{df.na.fill({'price':0}).filter(col('price').isNull()).count()}")

print(f"Total Stores: {store_revenue.count()}")

Pipeline Validation:
Original Records: 1000
After Removing Duplicates: 950
Remaining NULL Prices: 0
Total Stores: 15


##### Observation
- Duplicate records were removed successfully.
- Missing values in the `price` column were replaced with **0**.
- Revenue for each transaction was calculated using **Price × Quantity**.
- The cleaned dataset was grouped by **store_id** to calculate the total revenue generated by each store.
- The final pipeline demonstrates a complete ETL workflow involving data cleaning, transformation, and aggregation.
